# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

### Libraries

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
from pathlib import Path
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

### Input

Change these variables to what you need. The locations must be a string with each continent separated by a comma and space. The serotypes and genotypes must be lists. The dates must be strings. 

In [2]:
# Dates and locations
locations = "Antarctica, North America, South America"
serotypes = ["H5N1"]
genotypes = ["A3"]
start_date = "11-01-2021"
end_date = "08-07-2026"

# Make a date range
date_range = start_date + "--" + end_date


### Paths

Make sure these paths fit your schema. Here is the general tree structure: </br>

* Avian Flu: This is where the code in this repository is housed. "references" is a subdirectory here.
* Avian Flu Files: This is a sister directory to the repository -- that is, the code repository and the input/output files both have the same parent directory, which is the "home" variable.
* Avian Flu Files/NCBI Virus: This is the NCBI Virus directory. There are three subdirectories here: "downloads", "temp", and "complete".
* NCBI Virus/downloads: input data
* NCBI Virus/temp: intermediate data created between input and output. A subdirectory is created for this specific date range.
* NCBI Virus/complete: output data

<img src="../Avian_Flu_Files/Presentations/ncbi_virus_file_tree.png" width="500" height="250" alt="NCBI Virus file tree structure">

In [3]:
# Paths

home = Path.home() / "OneDrive - National Institutes of Health/Documents/Virus_Evolution/"
avian_flu_files = home / "Avian_Flu_Files" # Avian flu files is a sister directory of the directory this code is housed in
references = home / "Avian_Flu" / "references"
downloads = Path.home() / "Downloads"

downloads_saved = avian_flu_files / "NCBI_Virus/downloads" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))  

temp_files = avian_flu_files / "NCBI_Virus/temp" / ("segments_" + date_range)
if not temp_files.exists():
    Path.mkdir(temp_files, parents=True, exist_ok=True)
complete_files = avian_flu_files / "NCBI_Virus/complete" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))
if not complete_files.exists(): # checking if the directory exists or not
    Path.mkdir(complete_files, parents=True, exist_ok=True) # if the directory is not present then create it

states_ref = pd.read_csv(references / "states_ref.csv")

## Downloading Data

In [4]:
os.chdir(downloads)

if not downloads_saved.exists(): # checking if the directory exists or not
    Path.mkdir(downloads_saved, parents=True, exist_ok=True) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0: # If we have downloaded files already, skip
        break 
    else: # If we don't have any downloaded files, get files
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [5]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Assembly, as_index=False).size()
print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Assembly")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_counted.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments = metadata_complete_segs
metadata_segments

167143
              Assembly  size
0      GCA_038929245.1     8
1      GCA_038932225.1     8
2      GCA_038932275.1     8
3      GCA_038932295.1     8
4      GCA_038933705.1     8
...                ...   ...
19181  GCA_059952155.1     8
19182  GCA_059952165.1     8
19183  GCA_059952175.1     8
19184  GCA_059952205.1     8
19185  GCA_059952255.1     8

[19186 rows x 2 columns]


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153467,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153468,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153469,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153470,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [6]:
os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PZ803346.1 |Influenza A virus (A/Canada Goose...
1         >PZ803347.1 |Influenza A virus (A/Canada Goose...
2         >PZ803348.1 |Influenza A virus (A/Canada Goose...
3         >PZ803349.1 |Influenza A virus (A/Canada Goose...
4         >PZ803350.1 |Influenza A virus (A/Canada Goose...
                                ...                        
167138    >OK205883.1 |Influenza A virus (A/chicken/Vera...
167139    >OK205884.1 |Influenza A virus (A/chicken/Vera...
167140    >OK205885.1 |Influenza A virus (A/chicken/Vera...
167141    >OK205886.1 |Influenza A virus (A/chicken/Vera...
167142    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 167143, dtype: object
167143
151960


In [7]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803360.1 |Influenza A virus (A/Cackling Goo...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803361.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAA...
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803362.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGGTGACAAAAACATAATGGATTCCAACACTGTGTCAAGC...
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803363.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTACTTTT...
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803364.1 |Influenza A virus (A/Cackling Goo...,ATGAATCCAAATCAAAAGATAACAACTATCGGGTCAATCTGCATGG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151955,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...
151956,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...
151957,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...
151958,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...


## Find genotypes

Set up files that are friendly to multi-genoflu, then run multi-genoflu.

In [8]:
metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

print(len(metadata_segments))

151960


### Create FASTA files of unknown genotypes 

In [9]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,Partial_Header_temp
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803360.1 |Influenza A virus (A/Cackling Goo...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,A_Cackling_Goose_North_Dakota_2740_2026
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803361.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAA...,A_Cackling_Goose_North_Dakota_2740_2026
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803362.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGGTGACAAAAACATAATGGATTCCAACACTGTGTCAAGC...,A_Cackling_Goose_North_Dakota_2740_2026
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803363.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTACTTTT...,A_Cackling_Goose_North_Dakota_2740_2026
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803364.1 |Influenza A virus (A/Cackling Goo...,ATGAATCCAAATCAAAAGATAACAACTATCGGGTCAATCTGCATGG...,A_Cackling_Goose_North_Dakota_2740_2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151955,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...,1864
151956,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...,1864
151957,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...,1864
151958,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,1864


In [10]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments["Partial_Header"] = metadata_segments["Partial_Header_temp"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name
metadata_segments = metadata_segments.dropna(subset="Partial_Header")

print(metadata_segments["Partial_Header"].values[0:5])

['>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026']


In [11]:
# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

0         >A_Cackling_Goose_North_Dakota_2740_2026
1         >A_Cackling_Goose_North_Dakota_2740_2026
2         >A_Cackling_Goose_North_Dakota_2740_2026
3         >A_Cackling_Goose_North_Dakota_2740_2026
4         >A_Cackling_Goose_North_Dakota_2740_2026
                            ...                   
151955                                       >1864
151956                                       >1864
151957                                       >1864
151958                                       >1864
151959                                       >1864
Name: Partial_Header, Length: 151952, dtype: object
['>A_chicken_SK_FAV_1257_2_2022_2022' '>A_chicken_SK_FAV_1257_2_2022_2022'
 '>A_chicken_SK_FAV_1257_2_2022_2022' '>A_chicken_SK_FAV_1257_2_2022_2022'
 '>A_chicken_SK_FAV_1257_2_2022_2022' '>A_chicken_SK_FAV_1257_2_2022_2022'
 '>A_chicken_SK_FAV_1257_2_2022_2022' '>A_chicken_SK_FAV_1257_2_2022_2022']


### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To avoid job kill:
```
sinteractive
```
To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [12]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [12]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])

# Merge
metadata_genoflu = metadata_segments.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 
metadata_genoflu = metadata_genoflu.rename(columns={"Genotype_y":"Genotype", "Genotype_x":"Serotype"})

print(metadata_genoflu)


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_53472\3183371152.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_53472\3183371152.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)


         Accession GenBank_RefSeq         Assembly SRA_Accession BioSample  \
0       PZ803360.1        GenBank  GCA_059951935.1           NaN       NaN   
1       PZ803361.1        GenBank  GCA_059951935.1           NaN       NaN   
2       PZ803362.1        GenBank  GCA_059951935.1           NaN       NaN   
3       PZ803363.1        GenBank  GCA_059951935.1           NaN       NaN   
4       PZ803364.1        GenBank  GCA_059951935.1           NaN       NaN   
...            ...            ...              ...           ...       ...   
151883  OK205699.1        GenBank  GCA_039174385.1           NaN       NaN   
151884  OK205700.1        GenBank  GCA_039174385.1           NaN       NaN   
151885  OK205701.1        GenBank  GCA_039174385.1           NaN       NaN   
151886  OK205702.1        GenBank  GCA_039174385.1           NaN       NaN   
151887  OK205703.1        GenBank  GCA_039174385.1           NaN       NaN   

         BioProject      Organism_Name                         

In [13]:
# Cut down to only columns we want
metadata_genoflu = metadata_genoflu[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "Partial_Header_Merge"]] #, "Strain"]]

# Get genbank strain name
 
metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu = metadata_genoflu.dropna(subset="genbank_name") 

In [14]:
metadata_genoflu

# Make sure we only have the serotype(s) we want -- this code only works for a single serotype, so change it if multiple are ever needed
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

After running the below code, **STOP TO CHECK** if any new animals appear

In [15]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, write code dealing with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv", index=False) # Make sure name is different to avoid overwriting the first reference 


['fox', 'common grackle', 'house sparrow', 'ruddy turnstone', 'wyoming', 'american pelican', 'sterna hirundo', 'common murre', 'common goldeneye', 'american green-winged teal', 'hermit thrush', 'lesser scaup', 'american black duck', 'caspian tern', 'raccoon', 'american kestrel', 'white-winged scoter', 'pintail', 'northern shoveler', 'serval', 'owl', 'mergus', 'mottled duck', 'waterfowl', 'great shearwater', 'barn owl', 'striped skunk', 'megascops choliba', 'dolphin', 'rough-legged hawk', 'gadwell', 'northern harrier', 'rock dove', 'black skimmer', 'bovine', 'great blue heron', 'redhead duck', 'chukar partridge', 'colorado', 'harbor seal', 'finch', 'cat', 'peruvian booby', 'american woodcock', 'western grebe', 'pluvialis dominica', 'dunlin', 'brown pelican', 'procellaria aequinoctialis', 'wild goose', 'black-billed magpie', 'northwestern crow', 'laridae', 'black-legged kittiwake', 'vulpes vulpes', 'black-crowned night heron', 'lion', 'broad-winged hawk', 'burrowing owl', 'crested caraca

In [16]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [23]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype"] = metadata_genoflu["Genotype"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"

# Cut down dataframe only to genotypes we want

metadata_genoflu = metadata_genoflu[metadata_genoflu['Genotype'].isin(genotypes)]


In [24]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,Partial_Header_Merge,genbank_name,Host_Type,Years,Geo_Location_Normalized,Geo_Location_Country,Geo_Location_State_med
5128,PZ479108.1,GCA_058017755.1,Influenza A virus (A/Common Eider/AK/26G06093-...,common eider,2026-04-08,SRR38343085,A/Common Eider/AK/26G06093-001-original/2026,A3,USA: AK,>PZ479108.1 |Influenza A virus (A/Common Eider...,ATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAGTCTCGCA...,H5N1,1,A_Common_Eider_AK_26G06093_001_original_2026,A/Common Eider/AK/26G06093-001-original/2026,wild_avian,2026,USA-AK,USA,AK
5129,PZ479109.1,GCA_058017755.1,Influenza A virus (A/Common Eider/AK/26G06093-...,common eider,2026-04-08,SRR38343085,A/Common Eider/AK/26G06093-001-original/2026,A3,USA: AK,>PZ479109.1 |Influenza A virus (A/Common Eider...,ATGGATGTCAATCCGACTTTACTTTTCTTAAAAGTGCCAGCGCAAG...,H5N1,2,A_Common_Eider_AK_26G06093_001_original_2026,A/Common Eider/AK/26G06093-001-original/2026,wild_avian,2026,USA-AK,USA,AK
5130,PZ479110.1,GCA_058017755.1,Influenza A virus (A/Common Eider/AK/26G06093-...,common eider,2026-04-08,SRR38343085,A/Common Eider/AK/26G06093-001-original/2026,A3,USA: AK,>PZ479110.1 |Influenza A virus (A/Common Eider...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,H5N1,3,A_Common_Eider_AK_26G06093_001_original_2026,A/Common Eider/AK/26G06093-001-original/2026,wild_avian,2026,USA-AK,USA,AK
5131,PZ479111.1,GCA_058017755.1,Influenza A virus (A/Common Eider/AK/26G06093-...,common eider,2026-04-08,SRR38343085,A/Common Eider/AK/26G06093-001-original/2026,A3,USA: AK,>PZ479111.1 |Influenza A virus (A/Common Eider...,ATGGAGAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4,A_Common_Eider_AK_26G06093_001_original_2026,A/Common Eider/AK/26G06093-001-original/2026,wild_avian,2026,USA-AK,USA,AK
5132,PZ479112.1,GCA_058017755.1,Influenza A virus (A/Common Eider/AK/26G06093-...,common eider,2026-04-08,SRR38343085,A/Common Eider/AK/26G06093-001-original/2026,A3,USA: AK,>PZ479112.1 |Influenza A virus (A/Common Eider...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAGATGGAGACTG...,H5N1,5,A_Common_Eider_AK_26G06093_001_original_2026,A/Common Eider/AK/26G06093-001-original/2026,wild_avian,2026,USA-AK,USA,AK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139971,OQ959084.1,GCA_039296015.1,Influenza A virus (A/Bald Eagle/Alaska/22-0138...,bald eagle,2022-04-27,,A/Bald Eagle/Alaska/22-013831-001/2022,A3,USA: Alaska,>OQ959084.1 |Influenza A virus (A/Bald Eagle/A...,ATGGAGAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4,A_Bald_Eagle_Alaska_22_013831_001_2022,A/Bald Eagle/Alaska/22-013831-001/2022,wild_avian,2022,USA-Alaska,USA,Alaska
139972,OQ959085.1,GCA_039296015.1,Influenza A virus (A/Bald Eagle/Alaska/22-0138...,bald eagle,2022-04-27,,A/Bald Eagle/Alaska/22-013831-001/2022,A3,USA: Alaska,>OQ959085.1 |Influenza A virus (A/Bald Eagle/A...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAGATGGAGACTG...,H5N1,5,A_Bald_Eagle_Alaska_22_013831_001_2022,A/Bald Eagle/Alaska/22-013831-001/2022,wild_avian,2022,USA-Alaska,USA,Alaska
139973,OQ959086.1,GCA_039296015.1,Influenza A virus (A/Bald Eagle/Alaska/22-0138...,bald eagle,2022-04-27,,A/Bald Eagle/Alaska/22-013831-001/2022,A3,USA: Alaska,>OQ959086.1 |Influenza A virus (A/Bald Eagle/A...,ATGAATCCAAATCAAAGGATAATAACCACTGGATCAATCTGTATGG...,H5N1,6,A_Bald_Eagle_Alaska_22_013831_001_2022,A/Bald Eagle/Alaska/22-013831-001/2022,wild_avian,2022,USA-Alaska,USA,Alaska
139974,OQ959087.1,GCA_039296015.1,Influenza A virus (A/Bald Eagle/Alaska/22-0138...,bald eagle,2022-04-27,,A/Bald Eagle/Alaska/22-013831-001/2022,A3,USA: Alaska,>OQ959087.1 |Influenza A virus (A/Bald Eagle/A...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,A_Bald_Eagle_Alaska_22_013831_001_2022,A/Bald Eagle/Alaska/22-013831-001/2022,wild_avian,2022,USA-Alaska,USA,Alaska


In [25]:
def geo_location_normalize(geolocation: str) -> str:

    geolocation = geolocation.replace(":", ",") # We will need to split on commas later; e.g. USA: MD -> USA, MD

    country = geolocation.split(",")[0] # Get the first part of the geolocation, aka the country 

    state = geolocation.split(",")[-1] # Get the last part of the geolocation, aka the state

    state = state.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country = country.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country_state = country + "-" + state # Log needed

    return country_state 

# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

      
    metadata["Geo_Location_State_USA"] = metadata["Geo_Location_State_med"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name
    else x)
    

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_USA"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    return metadata

def geo_location_strain_name(metadata, state_ref_file):
    # If there is no state in the metadata, try the strain name
    state_ref = pd.read_csv(state_ref_file)

    metadata["strain_name_state_nonhuman"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2])

    metadata["strain_name_state_human"] = metadata["genbank_name"].apply(lambda x: x.split("/")[1])

    # If nonhuman, do strain name [2]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] != "human"), metadata["strain_name_state_nonhuman"], metadata["Geo_Location_State_USA"])

    # If human, do strain name [1]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] == "human"), metadata["strain_name_state_human"], metadata["Geo_Location_State_USA"])

    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name"].apply(geo_location_normalize)

    metadata["Geo_Location_State_Strain_Name_State"] = metadata["Geo_Location_State_Strain_Name"].apply(lambda x: x.split("-")[-1])
    
    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name_State"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name, and is just USA
    else ""
    if x == "USA" or x == "United_States"
    
    else x)

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_Strain_Name"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata

    
    return metadata

In [26]:
print(states_ref[states_ref['State'].str.contains("New_York")]["Abbreviation"].values[0])

NY


In [27]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))



In [28]:
os.chdir(references)
metadata_genoflu = geo_location_get(metadata_genoflu, "states_ref.csv")

In [29]:
metadata_genoflu = geo_location_strain_name(metadata_genoflu, "states_ref.csv")

In [30]:
# os.chdir(downloads_saved)
# metadata_genoflu.to_csv("metadata_checkpoint.csv")

In [31]:
# If there is no SRA Accession, replace identifier with Assembly 
metadata_genoflu["Identifier"] = metadata_genoflu["Assembly"].apply(lambda x: x if metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0] == "" else metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0]) # np.where(metadata_genoflu['SRA_Accession'] != "", metadata_genoflu['SRA_Accession'], metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]))

print((metadata_genoflu[metadata_genoflu["Identifier"].str.contains("SRR")])) # metadata_genoflu[(metadata_genoflu["Identifier"].str.contains("GCA")) | 

         Accession         Assembly  \
5128    PZ479108.1  GCA_058017755.1   
5129    PZ479109.1  GCA_058017755.1   
5130    PZ479110.1  GCA_058017755.1   
5131    PZ479111.1  GCA_058017755.1   
5132    PZ479112.1  GCA_058017755.1   
...            ...              ...   
104507  PQ798057.1  GCA_046438465.1   
104508  PQ798058.1  GCA_046438465.1   
104509  PQ798059.1  GCA_046438465.1   
104510  PQ798060.1  GCA_046438465.1   
104511  PQ798061.1  GCA_046438465.1   

                                            GenBank_Title          Host  \
5128    Influenza A virus (A/Common Eider/AK/26G06093-...  common eider   
5129    Influenza A virus (A/Common Eider/AK/26G06093-...  common eider   
5130    Influenza A virus (A/Common Eider/AK/26G06093-...  common eider   
5131    Influenza A virus (A/Common Eider/AK/26G06093-...  common eider   
5132    Influenza A virus (A/Common Eider/AK/26G06093-...  common eider   
...                                                   ...           ...   
104507

In [32]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_New"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [33]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
# metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


3376
3376


## Rename segments and make complete FASTA files

In [34]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

A3_PB2
A3_PB1
A3_PA
A3_HA
A3_NP
A3_NA
A3_MP
A3_NS


In [35]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files / (file_name), "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

         Accession         Assembly  \
5128    PZ479108.1  GCA_058017755.1   
5248    PZ479252.1  GCA_058017515.1   
5256    PZ479260.1  GCA_058017525.1   
5264    PZ479268.1  GCA_058017535.1   
5280    PZ479284.1  GCA_058018025.1   
...            ...              ...   
135408  PP802851.1  GCA_039823905.1   
135776  PP761806.1  GCA_039640125.1   
139312  OR267255.1  GCA_039320585.1   
139960  OQ959073.1  GCA_039295575.1   
139968  OQ959081.1  GCA_039296015.1   

                                            GenBank_Title  \
5128    Influenza A virus (A/Common Eider/AK/26G06093-...   
5248    Influenza A virus (A/Great Blue Heron/WA/26G06...   
5256    Influenza A virus (A/Great Blue Heron/WA/26G06...   
5264    Influenza A virus (A/Great Blue Heron/WA/26G06...   
5280    Influenza A virus (A/Herring Gull/CA/26G06613-...   
...                                                   ...   
135408  Influenza A virus (A/northern pintail/USA/IZ22...   
135776  Influenza A virus (A/black-legged k